In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

**Preprocessing**

1. Drop the 3 columns with the most NaN values

In [ ]:
df = pd.read_csv('GermanCredit.csv')

# Drop n cols with the most NaN vals
def dropn(n, df):
    assert(n < df.shape[1])

    res = df.isnull().sum(axis=0)
    isnone = lambda x: x == 'none'
    res += df.map(isnone).sum(axis=0)

    for i in range(n):
        max_col = res.idxmax(axis=0)
        res.drop(max_col, inplace=True)
        df.drop(max_col, axis=1, inplace=True)
        
dropn(3, df)
df.head(10)

2. Remove apostrophes

In [ ]:
remove_apostrophes = lambda x: str(x).replace("'", "")
df = df.map(remove_apostrophes)
df.head(10)

3. Replace checking_status column values

In [ ]:
def replace_col_vals(df, col, targets, replacements):
    df[col] = df[col].replace(targets, replacements)

replace_col_vals(df, 'checking_status', ['no checking', '<0', '0<=X<200', '>=200'], ['No Checking', 'Low', 'Medium', 'High'])
df.head(10)

4. Replace savings_status column values

In [ ]:
replace_col_vals(df, 'savings_status',['no known savings', '<100', '100<=X<500', '500<=X<1000', '>=1000'], ['No Savings', 'Low', 'Medium', 'High', 'High'])
df.head(10)

5. Replace class column values

In [ ]:
replace_col_vals(df, 'class', ['good', 'bad'], ['1', '0'])
df.head(10)

6. Replace employment column values

In [ ]:
replace_col_vals(df, 'employment', ['unemployed', '<1', '1<=X<4', '4<=X<7', '>=7'], ['Unemployed', 'Amateur', 'Professional', 'Experienced', 'Expert'])
df.head(10)

**Analysis**

1. Find correlations between categories:
    - Count each category of foreign workers for each class of credit.
    - Count each category of employment for each category of savings_status.

In [ ]:
credit_counts = pd.crosstab(df['foreign_worker'], df['credit_amount'])
savings_counts = pd.crosstab(df['employment'], df['savings_status'])
print(credit_counts)
print(savings_counts)

2. Find the average credit_amount of single males that have 4<=X<7 years of employment.

In [ ]:
# Find average of target column of group satisfying vals == column values
def avg_group(df, target, cols, vals):
    group = df.copy()
    for c, v in zip(cols, vals):
        group = group[group[c] == v]
    group = group[target]
    group = group.astype(float)
    return group.mean()
    
avg_credit = avg_group(df, 'credit_amount', ['personal_status', 'employment'], ['male single', 'Experienced'])
print(avg_credit)

3. Find the average credit duration for each of the job types.

In [ ]:
jobs = df['job'].value_counts()
for j in jobs.index:
    avg_credit = avg_group(df, 'duration', ['job'], [j])
    print('Average credit duration for', j, ':', avg_credit)

4. Find the most common checkings and savings status for the purpose 'education'.

In [ ]:
# Find mode of target column of group satisfying vals == column values
def mode_group(df, target, cols, vals):
    group = df.copy()
    for c, v in zip(cols, vals):
        group = group[group[c] == v]
    group = group[target]
    group_counts = group.value_counts()
    return group_counts.idxmax()

top_checking = mode_group(df, 'checking_status', ['purpose'], ['education'])
top_saving = mode_group(df, 'savings_status', ['purpose'], ['education'])
test = mode_group(df, "personal_status", ['savings_status'], ['No Savings'])
print(test)
print('Most common checking status:', top_checking)
print('Most common savings status:', top_saving)

**Visualization**
1. Plot subplots of two bar charts: one for savings_status (x-axis) to personal status (y-axis), and another for checking_status (x-axis) to personal_status (y-axis). In each of the charts, each personal status category bar (number of people in that category) should be of a different color.

In [ ]:
# Plot subplots of every value in col x to values of col y
def hist_subplots(df, x, y):
    x_categories = df[x].value_counts()
    y_categories = df[y].value_counts()
    size_x = len(x_categories)
    size_y = len(y_categories)
    colors = random.sample(list(mcolors.TABLEAU_COLORS.values()), size_y)
    fig, axes = plt.subplots(1, size_x, sharex=True)

    for i, cat in zip(range(size_x), x_categories.index):
        group = df[df[x] == cat][y]
        _, _, patches = axes[i].hist(group, bins=size_y)
        axes[i].set_title(cat)
        axes[i].set_xticks(range(size_y))
        for label in axes[i].get_xticklabels():
            label.set_rotation(45)
            label.set_ha('right')
        # Color bars
        for patch, color in zip(patches, colors):
            patch.set_facecolor(color)
    plt.tight_layout()
    plt.show()

# Savings x, personal status y
hist_subplots(df, 'savings_status', 'personal_status')
# Checking x, personal status y
hist_subplots(df, 'checking_status', 'personal_status')

2. For people having credit_amount more than 4000, plot a bar graph which maps property_magnitude (x-axis) to the average customer age for that magnitude (y-axis).

In [ ]:
def bar_subplots(df, x, y):
    x_categories = df[x].value_counts()
    size_x = len(x_categories)
    fig, axes = plt.subplots(1, size_x)
    bins = np.histogram_bin_edges(df[y].astype(float), bins='auto')

    for i, cat in zip(range(size_x), x_categories.index):
        group = df[df[x] == cat][y]
        group = group.astype(float)
        axes[i].hist(group, bins=bins)
        axes[i].set_title(cat)
    plt.tight_layout()
    plt.show()

filter = [True if float(n) > 4000 else False for n in df['credit_amount']]
group = df[filter]
bar_subplots(group, 'property_magnitude', 'age')

3. For people with a "High" savings_status and age above 40, use subplots to plot the following pie charts: personal status, credit history, job

In [ ]:
def pie_subplots(df, features):
    n = len(features)
    fig, axes = plt.subplots(1, n)

    for i, f in zip(range(n), features):
        data = df[f].value_counts()
        axes[i].pie(data, radius=1.2)
        axes[i].set_title(f)
        axes[i].legend(labels=data.index, loc='upper center', bbox_to_anchor=(0.5, -0.1))
    plt.tight_layout()
    plt.show()

group = df[df['savings_status'] == 'High']
filter = [True if float(n) > 40 else False for n in group['age']]
group = group[filter]
pie_subplots(group, ['personal_status', 'credit_history', 'job'])